In [ ]:
# === Common notebook preamble (load llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for locating the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates when running from the notebooks folder
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] failed to load the llm_math package: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === End preamble ===


# Ch 11. RNN and LSTM — The First Approach to Sequences

> **Learning Objectives**
> - Express the recurrent structure of an RNN mathematically
> - Understand long-term dependency problems and vanishing gradients
> - Understand why LSTM gate mechanisms mitigate these problems

## 11.1 Sequence Data and the Variable-Length Problem

Images are fixed-size vectors, but text is a variable-length sequence. How should we process it?

- **RNN**: process sequences with a recurrent structure
- **CNN (1D)**: capture local patterns with convolutions
- **Transformer (Ch 14+)**: model global dependencies with attention

This chapter covers RNNs and LSTMs. They were the standard before Transformers.

## 11.2 Mathematical Structure of an RNN

$$\mathbf{h}_t = \tanh(W_h \mathbf{h}_{t-1} + W_x \mathbf{x}_t + \mathbf{b})$$

- $\mathbf{x}_t \in \mathbb{R}^d$: input at time $t$
- $\mathbf{h}_t \in \mathbb{R}^h$: hidden state (memory) at time $t$
- $W_h \in \mathbb{R}^{h \times h}$, $W_x \in \mathbb{R}^{h \times d}$: parameters

Key idea: **reuse the same weights at every time step** (parameter sharing).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simple RNN cell (NumPy)
class RNNCell:
    def __init__(self, input_dim, hidden_dim, seed=0):
        rng = np.random.default_rng(seed)
        self.Wx = rng.standard_normal((input_dim, hidden_dim)) * 0.1
        self.Wh = rng.standard_normal((hidden_dim, hidden_dim)) * 0.1
        self.b = np.zeros(hidden_dim)
        self.hidden_dim = hidden_dim

    def forward(self, x_seq):
        """x_seq: (T, input_dim) -> h_seq: (T, hidden_dim)"""
        T = x_seq.shape[0]
        h = np.zeros(self.hidden_dim)
        h_seq = []
        for t in range(T):
            h = np.tanh(x_seq[t] @ self.Wx + h @ self.Wh + self.b)
            h_seq.append(h.copy())
        return np.array(h_seq)

    def last_hidden(self, x_seq):
        return self.forward(x_seq)[-1]

# Test
rnn = RNNCell(input_dim=4, hidden_dim=8)
x_seq = np.random.randn(10, 4)  # Length-10 sequence
h_seq = rnn.forward(x_seq)
print(f"Input sequence: {x_seq.shape}")
print(f"Hidden-state sequence: {h_seq.shape}")
print(f"Last hidden state: {h_seq[-1].round(4)}")


## 11.3 BPTT (Backpropagation Through Time)

RNN backpropagation = backpropagation through a computation graph unrolled along the time axis.

$$\frac{\partial \mathcal{L}}{\partial \mathbf{h}_0} = \prod_{t=1}^{T} \frac{\partial \mathbf{h}_t}{\partial \mathbf{h}_{t-1}} \cdot \frac{\partial \mathcal{L}}{\partial \mathbf{h}_T}$$

$\frac{\partial \mathbf{h}_t}{\partial \mathbf{h}_{t-1}} = \mathrm{diag}(1 - \mathbf{h}_t^2) W_h$ (including the derivative of tanh).

Problem: $\mathrm{diag}(1 - \mathbf{h}_t^2) \leq 1$. If the absolute eigenvalues of $W_h$ are less than 1, gradients **vanish**; if they are larger, gradients **explode**.


In [ ]:
# Simulate vanishing/exploding gradients
def simulate_rnn_gradient(T, Wh_norm, seed=0):
    """Gradient magnitude with respect to h_0 in a T-step RNN."""
    rng = np.random.default_rng(seed)
    h = rng.standard_normal(8)
    grad = np.ones(8)  # dL/dh_T
    for t in range(T):
        # Wh is a scaled identity matrix
        Wh = np.eye(8) * Wh_norm
        h_prev = rng.standard_normal(8)
        h = np.tanh(h_prev @ Wh)
        # Backward Pass: grad = grad @ Wh @ diag(1 - h^2)
        diag = 1 - h**2
        grad = grad @ Wh * diag
    return np.linalg.norm(grad)

print("RNN vanishing/exploding gradient simulation:")
print(f"{'T':>5} | {'||Wh||=0.5 (vanishing)':>20} | {'||Wh||=1.0':>15} | {'||Wh||=1.5 (exploding)':>20}")
print("-" * 65)
for T in [5, 10, 20, 50, 100]:
    g_small = simulate_rnn_gradient(T, 0.5)
    g_one = simulate_rnn_gradient(T, 1.0)
    g_big = simulate_rnn_gradient(T, 1.5)
    print(f"{T:>5} | {g_small:>20.4e} | {g_one:>15.4e} | {g_big:>20.4e}")

print("\n=> If ||Wh||<1, gradients decrease exponentially (vanishing)")
print("=> If ||Wh||>1, gradients increase exponentially (exploding)")


## 11.4 LSTM — Gate Mechanisms

LSTM (Hochreiter & Schmidhuber, 1997) controls gradient flow with gates.

$$\mathbf{f}_t = \sigma(W_f [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_f) \quad \text{(forget gate)}$$
$$\mathbf{i}_t = \sigma(W_i [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_i) \quad \text{(input gate)}$$
$$\tilde{\mathbf{c}}_t = \tanh(W_c [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_c) \quad \text{(candidate)}$$
$$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t \quad \text{(cell state)}$$
$$\mathbf{o}_t = \sigma(W_o [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_o) \quad \text{(output gate)}$$
$$\mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t) \quad \text{(hidden state)}$$

Key idea: **the cell state $\mathbf{c}_t$ is updated additively** → gradients flow more easily.


In [ ]:
# Implement an LSTM cell (NumPy)
class LSTMCell:
    def __init__(self, input_dim, hidden_dim, seed=0):
        rng = np.random.default_rng(seed)
        # Four gates: forget, input, candidate, output
        self.W = rng.standard_normal((input_dim + hidden_dim, 4 * hidden_dim)) * 0.1
        self.b = np.zeros(4 * hidden_dim)
        self.hidden_dim = hidden_dim

    def forward(self, x_seq):
        T = x_seq.shape[0]
        h = np.zeros(self.hidden_dim)
        c = np.zeros(self.hidden_dim)
        h_seq, c_seq = [], []
        for t in range(T):
            # Concatenate: [h, x]
            hx = np.concatenate([h, x_seq[t]])
            gates = hx @ self.W + self.b
            f, i, g, o = np.split(gates, 4)
            f = 1 / (1 + np.exp(-f))  # sigmoid
            i = 1 / (1 + np.exp(-i))
            g = np.tanh(g)
            o = 1 / (1 + np.exp(-o))
            c = f * c + i * g  # cell state (additive update!)
            h = o * np.tanh(c)
            h_seq.append(h.copy()); c_seq.append(c.copy())
        return np.array(h_seq), np.array(c_seq)

# Test
lstm = LSTMCell(input_dim=4, hidden_dim=8)
h_seq, c_seq = lstm.forward(x_seq)
print(f"LSTM Output: h_seq {h_seq.shape}, c_seq {c_seq.shape}")
print(f"Last h: {h_seq[-1].round(4)}")
print(f"Last c: {c_seq[-1].round(4)}")


## 11.5 Character-Level Language Model

Let's generate text with a simple character-level RNN/LSTM.


In [ ]:
# Character-level LSTM language model (PyTorch)
import torch
import torch.nn as nn

# Small Tiny Shakespeare sample
from llm_math.data import load_tiny_shakespeare
text = load_tiny_shakespeare(max_chars=20000)
chars = sorted(set(text))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}
vocab_size = len(chars)
print(f"Text length: {len(text)}, vocabulary size: {vocab_size}")
print(f"Characters: {chars[:20]}...")

# Prepare data
seq_len = 50
data = np.array([char_to_idx[c] for c in text])

def make_batch(data, seq_len, batch_size=32):
    """Generate a random batch."""
    n = len(data) - seq_len - 1
    starts = np.random.randint(0, n, batch_size)
    X = np.array([data[s:s+seq_len] for s in starts])
    y = np.array([data[s+1:s+1+seq_len] for s in starts])
    return torch.tensor(X, dtype=torch.long), torch.tensor(y, dtype=torch.long)

# Model
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, n_layers=1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, n_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        emb = self.embed(x)  # (B, T, E)
        out, hidden = self.lstm(emb, hidden)  # (B, T, H)
        logits = self.fc(out)  # (B, T, V)
        return logits, hidden

model = CharLSTM(vocab_size, embed_dim=32, hidden_dim=64)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nNumber of model parameters: {n_params:,}")


In [ ]:
# Training
import time

loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.003)

n_steps = 300
losses = []
t0 = time.time()
for step in range(n_steps):
    X, y = make_batch(data, seq_len, batch_size=32)
    opt.zero_grad()
    logits, _ = model(X)
    # (B*T, V) vs (B*T,)
    loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
    opt.step()
    losses.append(loss.item())
    if (step + 1) % 50 == 0:
        print(f"Step {step+1}: loss={loss.item():.4f}")
print(f"\nTraining time: {time.time() - t0:.1f}s")


In [ ]:
# Text generation
def generate(model, seed_text, length=200, temperature=0.8):
    model.eval()
    chars_idx = [char_to_idx[c] for c in seed_text]
    hidden = None
    with torch.no_grad():
        x = torch.tensor([chars_idx], dtype=torch.long)
        for _ in range(length):
            logits, hidden = model(x, hidden)
            # Last time step
            logit = logits[0, -1] / temperature
            probs = torch.softmax(logit, dim=0)
            next_idx = torch.multinomial(probs, 1).item()
            chars_idx.append(next_idx)
            # Next input is only the last character
            x = torch.tensor([[next_idx]], dtype=torch.long)
    return ''.join(idx_to_char[i] for i in chars_idx)

print("=== Generated text (after training) ===\n")
print(generate(model, "First Citizen:\n", length=300))


In [ ]:
# Learning curve
plt.figure(figsize=(10, 4))
plt.plot(losses, alpha=0.3, color='blue')
# Moving average
window = 20
smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
plt.plot(range(window-1, len(losses)), smoothed, 'r-', linewidth=2, label='Moving average')
plt.xlabel('Step'); plt.ylabel('Loss')
plt.title('Character-level LSTM Learning Curve')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch11_char_lstm.png', dpi=100, bbox_inches='tight')
plt.show()


## 11.6 GRU — A Simplification of LSTM

GRU (Cho et al., 2014) simplifies LSTM to two gates:

$$\mathbf{z}_t = \sigma(W_z [\mathbf{h}_{t-1}, \mathbf{x}_t]) \quad \text{(update gate)}$$
$$\mathbf{r}_t = \sigma(W_r [\mathbf{h}_{t-1}, \mathbf{x}_t]) \quad \text{(reset gate)}$$
$$\tilde{\mathbf{h}}_t = \tanh(W [\mathbf{r}_t \odot \mathbf{h}_{t-1}, \mathbf{x}_t])$$
$$\mathbf{h}_t = (1 - \mathbf{z}_t) \odot \mathbf{h}_{t-1} + \mathbf{z}_t \odot \tilde{\mathbf{h}}_t$$

No separate cell state, so it is simpler. Performance is similar to LSTM.


## 11.7 Why Attention Emerged — The RNN Bottleneck

Limits of RNNs:
1. **Sequential processing**: cannot be parallelized → slow
2. **Long-term dependencies**: even LSTMs forget distant information
3. **Last hidden-state bottleneck**: all encoder information is compressed into one vector

**Attention** solves this bottleneck by letting the decoder refer to all encoder hidden states at every step. (Bahdanau et al., 2014)

This becomes the starting point for Transformers (Ch 14).

## 11.8 Key Takeaways

| Concept | Formula | Characteristics |
|---|---|---|
| RNN | $\mathbf{h}_t = \tanh(W_h \mathbf{h}_{t-1} + W_x \mathbf{x}_t)$ | Sequential, vanishing/exploding gradients |
| LSTM | Gates + cell state (addition) | Handles long-term dependencies |
| GRU | Two gates (simplified) | Similar to LSTM |
| BPTT | Backpropagation over time | $\prod \frac{\partial h_t}{\partial h_{t-1}}$ |

## Exercises

1. Train an RNN character by character on "hello world" and generate text.
2. Train LSTM and GRU on the same data and compare convergence speed.
3. Compare gradient magnitudes of RNN and LSTM for sequence lengths 10, 50, and 100.
4. Demonstrate experimentally that an RNN forgets the beginning of a long sequence.
5. Compare generated text from a character-level LSTM with temperatures 0.3, 0.8, and 1.5.

> Solutions: `solutions/ch11_solutions.ipynb`
